In [ ]:
import os
for v in ["OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS",
        "VECLIB_MAXIMUM_THREADS","NUMEXPR_NUM_THREADS"]:
  os.environ[v] = "1"

import sys
sys.path.insert(0, "/experiments/dgutowska/wavlm-ecapa/scripts")

CSV_PATH = "/experiments/dgutowska/wavlm-ecapa/analytics_paper_2.0/data_cp.csv"
OUT_PATH = "/experiments/dgutowska/wavlm-ecapa/analytics_paper_2.0/features_pool_NEW.parquet"
N_PROC   = 16
SEC      = 3.0

In [ ]:
def extract_one(args):
  idx, wav_path = args
  try:
      from extract_n import extract_n_sec
      feats = extract_n_sec(wav_path, SEC)
      return (idx, feats, None)
  except Exception as e:
      return (idx, None, f"{type(e).__name__}: {e}")


In [ ]:
import pandas as pd

df = pd.read_csv(CSV_PATH)
df["wav_path"] = df["dest"].str.replace(r"\.m4a$", ".wav", regex=True)
print(f"rows: {len(df)}")
print(f"unique wav: {df['wav_path'].nunique()}")
print(df.head(2)[["VoxCeleb_ID","speaker_age","wav_path"]].to_string())


In [ ]:
import time, multiprocessing as mp

rows   = []
n_fail = 0
todo   = list(enumerate(df["wav_path"]))
total  = len(todo)
t0     = time.time()

In [ ]:
with mp.Pool(N_PROC) as pool:
  for i, (idx, feats, err) in enumerate(pool.imap_unordered(extract_one, todo, chunksize=4)):
      if err:
          n_fail += 1
          print(f"FAIL row {idx}: {err}")
      else:
          feats["__row_idx"] = idx
          rows.append(feats)

      if (i + 1) % 200 == 0:
          dt   = time.time() - t0
          rate = (i + 1) / dt
          eta  = (total - i - 1) / rate
          print(f"[{i+1:6d}/{total}] fail={n_fail} "
                f"rate={rate:5.1f} f/s elapsed={dt/60:5.1f}min eta={eta/60:5.1f}min")

print(f"\n[done] ok={len(rows)} failed={n_fail} wall={(time.time()-t0)/60:.1f}min")


In [ ]:
feats_df = pd.DataFrame(rows).set_index("__row_idx").sort_index()
META = ["VoxCeleb_ID", "video_id", "gender", "speaker_age", "wav_path"]
final = df[META].join(feats_df, how="inner")
final.to_parquet(OUT_PATH, compression="zstd")
print(f"saved: {OUT_PATH}")
print(f"shape: {final.shape}")

In [39]:
import numpy as np

In [1]:
import pandas as pd

In [5]:
data = pd.read_pickle("selected_speakers_id_by_age_range.pkl")

In [6]:
import pandas as pd
from pathlib import Path

ROOT = Path("/experiments/dgutowska/wavlm-ecapa/analytics_paper_2.0")

# ── pełny dataset (47,764 wierszy)
df_pool = pd.read_parquet(ROOT / "features_pool.parquet")
print("pool:", df_pool.shape)   # (47764, 470)

# ── 1 nagranie / speaker (6,554 wierszy) z subset train/test
df_one = pd.read_parquet(ROOT / "features_one_per_speaker.parquet")
print("one_per_speaker:", df_one.shape)   # (6554, 470)
print(df_one["subset"].value_counts())

#Wczytanie wybranych kolumn (szybsze)

# Same metadane (bez 464 cech)
meta_cols = ["VoxCeleb_ID", "video_id", "gender", "speaker_age", "subset", "wav_path"]
df_meta = pd.read_parquet(ROOT / "features_one_per_speaker.parquet", columns=meta_cols)

# Tylko cechy MFCC + meta
mfcc_cols = [f"mfcc{i:02d}_mean" for i in range(1, 14)]      # baseline
mfcc_aux  = [f"mean_mfcc{i}" for i in range(1, 21)]           # AUX paper-style
df_mfcc = pd.read_parquet(ROOT / "features_pool.parquet",
                        columns=meta_cols[:5] + mfcc_cols + mfcc_aux)



pool: (47764, 470)
one_per_speaker: (6554, 470)
subset
train    3825
test     2729
Name: count, dtype: int64


In [ ]:
train

In [7]:
data.shape

(10932, 6)

In [12]:
features=df_pool[df_pool.VoxCeleb_ID.isin(data.VoxCeleb_ID)]

In [13]:
features.shape

(10932, 470)

# korelacja cech z wiekiem

In [ ]:
META=['VoxCeleb_ID', 'video_id', 'gender', 'speaker_age', 'wav_path',
       'subset']
FEATS = [c for c in features.columns if c not in META]


In [16]:
META=['VoxCeleb_ID', 'video_id', 'gender', 'speaker_age', 'wav_path',
       'subset']

In [19]:
FEATS = [c for c in features.columns if c not in META]


In [20]:
FEATS[:20]

['jitter_local',
 'jitter_local_abs',
 'jitter_rap',
 'jitter_ppq5',
 'jitter_ddp',
 'shimmer_local',
 'shimmer_apq3',
 'shimmer_apq5',
 'shimmer_apq11',
 'shimmer_dda',
 'shimmer_local_dB',
 'hnr_mean',
 'hnr_std',
 'F1_mean',
 'F1_std',
 'F1_median',
 'F2_mean',
 'F2_std',
 'F2_median',
 'F3_mean']

In [23]:
repr(features.speaker_age.values[0])

'np.float64(63.0)'

In [24]:
# Pearson — domyślnie (liniowa)
corr_pearson = features[FEATS].corrwith(features["speaker_age"])


In [25]:

# Top 20 wg wartości bezwzględnej
corr_results = corr_pearson.abs().sort_values(ascending=False)



In [31]:
corr = pd.DataFrame({
"pearson":  features[FEATS].corrwith(features["speaker_age"], method="pearson"),
"spearman": features[FEATS].corrwith(features["speaker_age"], method="spearman"),
"kendall":  features[FEATS].corrwith(features["speaker_age"], method="kendall"),  # ~30s dla 6554
})
corr["abs_pearson"] = corr["pearson"].abs()
corr["abs_spearman"] = corr["spearman"].abs()
corr["abs_kendall"] = corr["kendall"].abs()
top30_pearson  = corr.sort_values("abs_pearson",  ascending=False).head(30)
top30_spearman = corr.sort_values("abs_spearman", ascending=False).head(30)
top30_kendall  = corr.sort_values("abs_kendall",  ascending=False).head(30)


In [40]:
np.sort(top30_pearson.reset_index().rename(columns={"index": "feature"}).feature)

array(['F1_std', 'F2_std', 'F3_std', 'ber_0_500', 'ber_500_1000',
       'cpp_mean', 'cpps_mean', 'd_mfcc01_std', 'dd_mfcc01_std',
       'f0_mean', 'jitter_local', 'jitter_local_abs', 'loudness_db_range',
       'loudness_db_std', 'lpc_res_hnr_mean', 'mean_mfcc13', 'mean_mfcc9',
       'mfcc01_std', 'mfcc02_std', 'mfcc09_mean', 'mfcc13_mean',
       'std_mfcc1', 'std_mfcc2', 'std_mfcc_delta2_1', 'std_mfcc_delta_1',
       'syllable_count', 'syllable_rate', 'var_mfcc1',
       'var_mfcc_delta2_1', 'var_mfcc_delta_1'], dtype=object)

In [41]:
np.sort(top30_spearman.reset_index().rename(columns={"index": "feature"}).feature)

array(['F1_std', 'F2_std', 'F3_std', 'ber_0_500', 'cpp_mean', 'cpps_mean',
       'd_mfcc01_std', 'dd_mfcc01_std', 'f0_mean', 'jitter_local',
       'jitter_local_abs', 'loudness_db_range', 'loudness_db_std',
       'lpc_res_hnr_mean', 'mean_mfcc13', 'mean_mfcc9', 'mfcc01_std',
       'mfcc02_std', 'mfcc09_mean', 'mfcc13_mean', 'std_mfcc1',
       'std_mfcc2', 'std_mfcc_delta2_1', 'std_mfcc_delta_1',
       'syllable_count', 'syllable_rate', 'var_mfcc1', 'var_mfcc2',
       'var_mfcc_delta2_1', 'var_mfcc_delta_1'], dtype=object)

In [42]:
np.sort(top30_kendall.reset_index().rename(columns={"index": "feature"}).feature)

array(['F1_std', 'F2_std', 'F3_std', 'ber_0_500', 'cpp_mean', 'cpps_mean',
       'd_mfcc01_std', 'd_mfcc02_std', 'dd_mfcc01_std', 'f0_mean',
       'jitter_local', 'jitter_local_abs', 'loudness_db_std',
       'lpc_res_hnr_mean', 'mean_mfcc13', 'mean_mfcc9', 'mfcc01_std',
       'mfcc02_std', 'mfcc09_mean', 'mfcc13_mean', 'std_mfcc1',
       'std_mfcc2', 'std_mfcc_delta2_1', 'std_mfcc_delta_1',
       'syllable_count', 'syllable_rate', 'var_mfcc1', 'var_mfcc2',
       'var_mfcc_delta2_1', 'var_mfcc_delta_1'], dtype=object)

In [45]:


top_p = set(corr.nlargest(30, "abs_pearson").index)
top_s = set(corr.nlargest(30, "abs_spearman").index)
top_k = set(corr.nlargest(30, "abs_kendall").index)

important = top_p | top_s | top_k       
print(f"Łącznie unikatów: {len(important)}")  

Łącznie unikatów: 32


In [48]:
features[FEATS + ["speaker_age"]].corr()

,jitter_local,jitter_local_abs,jitter_rap,jitter_ppq5,jitter_ddp,shimmer_local,shimmer_apq3,shimmer_apq5,shimmer_apq11,shimmer_dda,...,std_spectral_rolloff,var_spectral_rolloff,mean_spectral_flatness,std_spectral_flatness,var_spectral_flatness,mean_zcr,std_zcr,var_zcr,tempo,speaker_age
jitter_local,1.000000,0.801459,0.921725,0.922849,0.921725,0.541744,0.462296,0.508199,0.442094,0.462296,...,0.146792,0.152743,-0.080243,-0.098678,-0.062477,-0.252322,-0.058435,-0.050021,0.005884,0.230854
jitter_local_abs,0.801459,1.000000,0.662950,0.721561,0.662950,0.397030,0.302419,0.357524,0.343830,0.302419,...,0.210351,0.209202,-0.061471,-0.033584,-0.013257,-0.294312,0.008662,0.005632,0.012155,0.282236
jitter_rap,0.921725,0.662950,1.000000,0.936366,1.000000,0.594164,0.565144,0.582197,0.453478,0.565144,...,0.033670,0.045176,-0.140449,-0.189797,-0.134759,-0.251309,-0.143874,-0.122004,0.007016,0.105575
jitter_ppq5,0.922849,0.721561,0.936366,1.000000,0.936366,0.581028,0.510770,0.565600,0.495253,0.510770,...,0.070873,0.078630,-0.106439,-0.137900,-0.097238,-0.241160,-0.105936,-0.092064,0.006222,0.168908
jitter_ddp,0.921725,0.662950,1.000000,0.936366,1.000000,0.594164,0.565144,0.582197,0.453478,0.565144,...,0.033670,0.045176,-0.140449,-0.189797,-0.134759,-0.251309,-0.143874,-0.122004,0.007016,0.105575
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
mean_zcr,-0.252322,-0.294312,-0.251309,-0.241160,-0.251309,-0.190078,-0.228304,-0.222906,-0.124349,-0.228304,...,0.189922,0.202757,0.776880,0.597771,0.460848,1.000000,0.668013,0.645185,0.000920,-0.058143
std_zcr,-0.058435,0.008662,-0.143874,-0.105936,-0.143874,-0.272769,-0.332612,-0.301193,-0.162667,-0.332612,...,0.670938,0.670069,0.572723,0.591507,0.437179,0.668013,1.000000,0.968410,-0.015949,0.078052
var_zcr,-0.050021,0.005632,-0.122004,-0.092064,-0.122004,-0.244959,-0.296117,-0.269323,-0.151729,-0.296117,...,0.625033,0.641686,0.515614,0.508757,0.389094,0.645185,0.968410,1.000000,-0.018977,0.066975
tempo,0.005884,0.012155,0.007016,0.006222,0.007016,0.002821,0.010243,0.004219,-0.005100,0.010243,...,-0.014567,-0.015495,-0.001623,-0.010455,-0.012650,0.000920,-0.015949,-0.018977,1.000000,0.004279


In [47]:
important

{'F1_std',
 'F2_std',
 'F3_std',
 'ber_0_500',
 'ber_500_1000',
 'cpp_mean',
 'cpps_mean',
 'd_mfcc01_std',
 'd_mfcc02_std',
 'dd_mfcc01_std',
 'f0_mean',
 'jitter_local',
 'jitter_local_abs',
 'loudness_db_range',
 'loudness_db_std',
 'lpc_res_hnr_mean',
 'mean_mfcc13',
 'mean_mfcc9',
 'mfcc01_std',
 'mfcc02_std',
 'mfcc09_mean',
 'mfcc13_mean',
 'std_mfcc1',
 'std_mfcc2',
 'std_mfcc_delta2_1',
 'std_mfcc_delta_1',
 'syllable_count',
 'syllable_rate',
 'var_mfcc1',
 'var_mfcc2',
 'var_mfcc_delta2_1',
 'var_mfcc_delta_1'}

In [51]:
features.to_pickle("features.pkl")